# **Linear Regression & Regularization - Complete Guide**

## **1. Deriving the Analytical Solution**

### **Problem Setup**
We have:
- `n` data points
- `p` features
- Design matrix: `X ∈ ℝⁿˣᵖ` (with added column of 1s for intercept)
- Response vector: `y ∈ ℝⁿ`
- Coefficient vector: `β ∈ ℝᵖ`

**Goal:** Find β that minimizes squared error:
```
L(β) = ||y - Xβ||²
     = (y - Xβ)ᵀ(y - Xβ)
```

### **Derivation Steps**

#### **Step 1: Expand the loss function**
```
L(β) = yᵀy - 2βᵀXᵀy + βᵀXᵀXβ
```

#### **Step 2: Take derivative with respect to β**
```
∂L/∂β = -2Xᵀy + 2XᵀXβ
```

#### **Step 3: Set derivative to zero (for minimum)**
```
-2Xᵀy + 2XᵀXβ = 0
XᵀXβ = Xᵀy
```

#### **Step 4: Solve for β**
If `XᵀX` is invertible:
```
β̂ = (XᵀX)⁻¹Xᵀy
```

**This is the Ordinary Least Squares (OLS) solution.**

---

## **2. Adding L1 and L2 Regularization**

### **L2 Regularization (Ridge Regression)**
**Loss function:** `L(β) = ||y - Xβ||² + λ||β||²`

Where:
- `λ` = regularization strength
- `||β||²` = β₁² + β₂² + ... + βₚ² (sum of squared coefficients)

**Solution:**
```
∂L/∂β = -2Xᵀy + 2XᵀXβ + 2λβ = 0
(XᵀX + λI)β = Xᵀy
β̂_ridge = (XᵀX + λI)⁻¹Xᵀy
```

**What changes:**
- `λI` is added to `XᵀX` → always invertible (no singularity)
- Coefficients are **shrunk toward zero**
- **All coefficients remain non-zero**

### **L1 Regularization (Lasso Regression)**
**Loss function:** `L(β) = ||y - Xβ||² + λ||β||₁`

Where:
- `||β||₁` = |β₁| + |β₂| + ... + |βₚ| (sum of absolute values)

**Solution:**
No closed-form solution! Must use optimization methods:
- Coordinate descent
- Least angle regression (LARS)
- Proximal gradient methods

**What changes:**
- Some coefficients become **exactly zero**
- Performs **automatic feature selection**
- Creates **sparse solutions**

---

## **3. Why L1 Regularization Selects Features**

### **Geometric Intuition**
**L2 constraint:** Circle/sphere boundary
- Coefficients can be anywhere on/inside the sphere
- **All** coefficients typically non-zero

**L1 constraint:** Diamond boundary
- Corners of the diamond lie on axes
- Solution often at a corner → some coefficients = 0

### **Mathematical Explanation**
The absolute value in L1 has a **kink at zero**:
- Gradient is ±1 (depending on sign of β)
- At β=0, gradient can push coefficient to exactly zero and keep it there

### **Feature Selection Process**
1. **Soft thresholding:** Coefficients below λ/2 are set to zero
2. **Automatic selection:** Unimportant features get zero weight
3. **Model simplification:** Only important features remain

### **Why Many Weights = 0**
```
For Lasso, the optimality condition is:
|Xⱼᵀ(y - Xβ)| ≤ λ/2  for all j

If this holds with strict inequality for feature j:
Then βⱼ = 0
```

**In practice:**
- Features with weak relationship to y get β=0
- Redundant features get β=0
- Only strongest predictors remain

---

## **4. Making Linear Models Nonlinear**

### **Method 1: Basis Function Expansion**
Transform features using nonlinear functions:

#### **Polynomial Features**
```
Original: x
Transformed: [1, x, x², x³, ..., xᵈ]
```
**Example:** Degree 2 polynomial regression
```
ŷ = β₀ + β₁x + β₂x²
```

#### **Radial Basis Functions (RBF)**
```
φⱼ(x) = exp(-γ||x - μⱼ||²)
```
Creates localized "bumps" in feature space.

#### **Spline Basis Functions**
Piecewise polynomials with smooth connections.

### **Method 2: Kernel Methods**
**Idea:** Work in high-dimensional space without explicitly computing it.

#### **Kernel Ridge Regression**
Use kernel matrix `K` where:
```
Kᵢⱼ = k(xᵢ, xⱼ)
```
Common kernels:
- Polynomial: `k(x,z) = (xᵀz + c)ᵈ`
- RBF: `k(x,z) = exp(-γ||x-z||²)`

**Solution:** `α = (K + λI)⁻¹y`

### **Method 3: Feature Engineering**
Create new features from existing ones:

#### **Interaction Terms**
```
x₁, x₂ → also include x₁·x₂
```

#### **Transformations**
```
log(x), exp(x), √x, 1/x
```

#### **Binning**
Convert continuous to categorical:
```
Age → [0-18], [19-30], [31-50], [51+]
```

### **Practical Implementation**

```python
# Example: Polynomial features + Ridge regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

# Create polynomial features up to degree 3
poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)

# Fit regularized linear model
model = Ridge(alpha=1.0)
model.fit(X_poly, y)

# Or using kernel ridge for RBF
from sklearn.kernel_ridge import KernelRidge
model = KernelRidge(kernel='rbf', alpha=1.0, gamma=0.1)
model.fit(X, y)
```

### **Key Considerations**

1. **Curse of dimensionality:** High-degree polynomials create many features
2. **Regularization essential:** To prevent overfitting in high-dimensional space
3. **Interpretability trade-off:** Nonlinear models are harder to interpret
4. **Computational cost:** Kernel methods scale as O(n³) or O(n²)

---

## **Summary Table**

| Method | Formula | Feature Selection | Nonlinear? |
|--------|---------|-------------------|------------|
| **OLS** | `β = (XᵀX)⁻¹Xᵀy` | No | No |
| **Ridge (L2)** | `β = (XᵀX + λI)⁻¹Xᵀy` | No (shrinkage) | No |
| **Lasso (L1)** | Numerical optimization | Yes (sparse) | No |
| **Polynomial+Ridge** | `β = (ΦᵀΦ + λI)⁻¹Φᵀy` | No | Yes |
| **Kernel Ridge** | `α = (K + λI)⁻¹y` | No | Yes |

---

## **When to Use What**

### **Use Linear/Ridge when:**
- Features already meaningful
- All features potentially relevant
- Want stable, interpretable model

### **Use Lasso when:**
- Many features, expect few to matter
- Need feature selection
- OK with some instability in selected features

### **Use Nonlinear methods when:**
- Relationships clearly curved
- Linear model underfits
- Have enough data to learn complexity

**Remember:** Always validate with train/test splits or cross-validation!

In [563]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np 
import seaborn as sns 
import os 
import re 
import lightgbm
import scipy
import statsmodels
from collections import Counter 

from sklearn.linear_model import Lasso , LinearRegression , Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , PolynomialFeatures , KernelCenterer
from sklearn.metrics import accuracy_score , mean_absolute_error , mean_absolute_percentage_error , mean_squared_error


In [564]:
def see(): 
 if os.path.exists('../datasets/train.json'):
  print("it is exististing dont panic ")
 else : 
  raise FileExistsError("basicaly it doesnt ecist you are on drugs ")
see()

it is exististing dont panic 


In [565]:
df_train = pd.read_json('../datasets/train.json')
df_test = pd.read_json('../datasets/train.json')

In [566]:
df_test.head()
df_test.describe


<bound method NDFrame.describe of         bathrooms  bedrooms                       building_id  \
4             1.0         1  8579a0b0d54db803821a35a4a615e97a   
6             1.0         2  b8e75fc949a6cd8225b455648a951712   
9             1.0         2  cd759a988b8f23924b5a2058d5ab2b49   
10            1.5         3  53a5b119ba8f7b61d4e010512e0dfc85   
15            1.0         0  bfb9405149bfff42a92980b594c28234   
...           ...       ...                               ...   
124000        1.0         3  92bbbf38baadfde0576fc496bd41749c   
124002        1.0         2  5565db9b7cba3603834c4aa6f2950960   
124004        1.0         1  67997a128056ee1ed7d046bbb856e3c7   
124008        1.0         2  3c0574a740154806c18bdf1fddd3d966   
124009        1.0         3  d89f514c3ed0abaae52cba7017ac0701   

                    created  \
4       2016-06-16 05:55:27   
6       2016-06-01 05:44:33   
9       2016-06-14 15:19:59   
10      2016-06-24 07:54:24   
15      2016-06-28 03:50:23   

In [567]:
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()
from collections import Counter 

df_train['interest_level'] = label.fit_transform(df_train['interest_level'])
df_train['interest_level']

4         2
6         1
9         2
10        2
15        1
         ..
124000    1
124002    2
124004    2
124008    2
124009    0
Name: interest_level, Length: 49352, dtype: int64

In [568]:
df_train['features'].apply(
    lambda lst : [re.sub(r'\s\'"','',f) for f in lst if isinstance(f,str)]
)
show = [f for x in df_train['features'] for f in x ]
show1 = Counter(show)


In [569]:
df_train['features'].to_frame().to_clipboard


<bound method NDFrame.to_clipboard of                                                  features
4       [Dining Room, Pre-War, Laundry in Building, Di...
6       [Doorman, Elevator, Laundry in Building, Dishw...
9       [Doorman, Elevator, Laundry in Building, Laund...
10                                                     []
15      [Doorman, Elevator, Fitness Center, Laundry in...
...                                                   ...
124000            [Elevator, Dishwasher, Hardwood Floors]
124002  [Common Outdoor Space, Cats Allowed, Dogs Allo...
124004  [Dining Room, Elevator, Pre-War, Laundry in Bu...
124008  [Pre-War, Laundry in Unit, Dishwasher, No Fee,...
124009  [Dining Room, Elevator, Laundry in Building, D...

[49352 rows x 1 columns]>

In [570]:
df_train['features'] = df_train['features'].apply(
    lambda lst : [re.sub(r'[\s\'"]','',f) for f in lst if isinstance(f,str)]
)

df_train = df_train[(df_train['price'] >= df_train['price'].quantile(0.01)) & (df_train['price'] <= df_train['price'].quantile(0.99))]

df_test['features'] = df_test['features'].apply(
    lambda x : [re.sub('r[\s\'"]', '', s)for s in x if isinstance(x,str)] 
)

df_test = df_test[(df_test['price'] >= df_test['price'].quantile(0.01)) & (df_test['price'] <= df_test['price'].quantile(0.99))]


In [571]:
df_train['features'].reset_index()

,index,features
0,4,"[DiningRoom, Pre-War, LaundryinBuilding, Dishw..."
1,6,"[Doorman, Elevator, LaundryinBuilding, Dishwas..."
2,9,"[Doorman, Elevator, LaundryinBuilding, Laundry..."
3,10,[]
4,15,"[Doorman, Elevator, FitnessCenter, LaundryinBu..."
...,...,...
48374,124000,"[Elevator, Dishwasher, HardwoodFloors]"
48375,124002,"[CommonOutdoorSpace, CatsAllowed, DogsAllowed,..."
48376,124004,"[DiningRoom, Elevator, Pre-War, LaundryinBuild..."
48377,124008,"[Pre-War, LaundryinUnit, Dishwasher, NoFee, Ou..."


In [572]:
all_features = [f for sub in df_train['features'] 
                    for f in sub ]
feature_count = Counter(all_features)
all_features
feature_count

Counter({'Elevator': 25398,
         'HardwoodFloors': 23159,
         'CatsAllowed': 23148,
         'DogsAllowed': 21662,
         'Doorman': 20497,
         'Dishwasher': 20095,
         'NoFee': 17806,
         'LaundryinBuilding': 16093,
         'FitnessCenter': 13000,
         'Pre-War': 8978,
         'LaundryinUnit': 8448,
         'RoofDeck': 6423,
         'OutdoorSpace': 5137,
         'DiningRoom': 4901,
         'HighSpeedInternet': 4225,
         'Balcony': 2898,
         'SwimmingPool': 2648,
         'LaundryInBuilding': 2565,
         'NewConstruction': 2507,
         'Terrace': 2179,
         'Exclusive': 2080,
         'Loft': 2042,
         'Garden/Patio': 1879,
         'WheelchairAccess': 1312,
         'CommonOutdoorSpace': 1280,
         'HARDWOOD': 880,
         'SIMPLEX': 873,
         'Fireplace': 837,
         'prewar': 820,
         'LOWRISE': 764,
         'Garage': 737,
         'LaundryRoom': 713,
         'ReducedFee': 686,
         'LaundryInUnit': 68

In [573]:
unique = set(feature_count)
len(unique)

1528

In [574]:
top20 = [feature[0] for feature in feature_count.most_common(21)]
top20

['Elevator',
 'HardwoodFloors',
 'CatsAllowed',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace',
 'Exclusive']

In [575]:
for feature in top20:
    df_train[feature] = df_train['features'].apply(lambda x: 1 if feature in x else 0)
    df_test[feature] = df_test['features'].apply(lambda x: 1 if feature in x else 0)

C:\Users\hosse\AppData\Local\Temp\ipykernel_47084\1600368820.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test[feature] = df_test['features'].apply(lambda x: 1 if feature in x else 0)
C:\Users\hosse\AppData\Local\Temp\ipykernel_47084\1600368820.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test[feature] = df_test['features'].apply(lambda x: 1 if feature in x else 0)
C:\Users\hosse\AppData\Local\Temp\ipykernel_47084\1600368820.py:3: SettingWithCopyWarning: 
A value is trying to be set on a

In [576]:
df_train


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,...,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,Exclusive
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[DiningRoom, Pre-War, LaundryinBuilding, Dishw...",40.7108,7170325,-73.9539,...,0,0,1,0,0,0,0,0,0,0
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, LaundryinBuilding, Dishwas...",40.7513,7092344,-73.9722,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, LaundryinBuilding, Laundry...",40.7575,7158677,-73.9625,...,0,0,0,0,0,0,0,0,0,0
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,...,0,0,0,0,0,0,0,0,0,0
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, FitnessCenter, LaundryinBu...",40.7439,7225292,-73.9743,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,1.0,3,92bbbf38baadfde0576fc496bd41749c,2016-04-05 03:58:33,There is 700 square feet of recently renovated...,W 171 Street,"[Elevator, Dishwasher, HardwoodFloors]",40.8433,6824800,-73.9396,...,0,0,0,0,0,0,0,0,0,0
124002,1.0,2,5565db9b7cba3603834c4aa6f2950960,2016-04-02 02:25:31,"2 bedroom apartment with updated kitchen, rece...",Broadway,"[CommonOutdoorSpace, CatsAllowed, DogsAllowed,...",40.8198,6813268,-73.9578,...,0,0,0,0,0,0,1,0,0,0
124004,1.0,1,67997a128056ee1ed7d046bbb856e3c7,2016-04-26 05:42:03,No Brokers Fee * Never Lived 1 Bedroom 1 Bathr...,210 Brighton 15th St,"[DiningRoom, Elevator, Pre-War, LaundryinBuild...",40.5765,6927093,-73.9554,...,0,0,1,0,0,0,0,0,0,0
124008,1.0,2,3c0574a740154806c18bdf1fddd3d966,2016-04-19 02:47:33,Wonderful Bright Chelsea 2 Bedroom apartment o...,West 21st Street,"[Pre-War, LaundryinUnit, Dishwasher, NoFee, Ou...",40.7448,6892816,-74.0017,...,0,1,0,0,0,0,0,0,0,0


In [577]:
df_final = ['bathrooms', 'bedrooms'] + top20
df_final

['bathrooms',
 'bedrooms',
 'Elevator',
 'HardwoodFloors',
 'CatsAllowed',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace',
 'Exclusive']

In [578]:
x_train  = df_train[df_final]
y_tarin = df_train['price']
X_test = df_train[df_final]
y_test = df_test['price']


In [579]:
class regression():
    def __init__(self,method ='sgd',learning_rate = 0.001,iteration= 100, seed= 21,batch_size = 32 ):
        self.learning_rate =learning_rate
        self.iteration= iteration
        self.seed = seed
        self.w = None
        self.method = method
        self.batch_size = batch_size
    def linearregression(self,x,y):
        #first we need to make our intercept and then we need to caucalte our rate
        w0 = x.shape[0]
        # with the c_ we target the intercept like if our x be ([2,3],[3,4]) after the c_(np.ones((w0,1)))we creat the 1 at first of our arrays as interecept or b  
        xb = np.c_[np.ones((w0,1)),x]
        # in here we basically count the w = (x . x.t)^-1 . x.t . y
        self.w = np.linalg.pinv(xb.T @ xb) @ xb.T @ y

    def fit_gb(self,x,y):
        np.random.seed(self.seed)
        w0 = x.shape[0]
        xb = np.c_[np.ones((w0,1)),x]
        self.w = np.zeros(xb.shape[1])
        n = xb.shape[0]
        for i in range(self.iteration):
            y_pred = xb @ self.w
            gred = (2/n) * (xb.T @ (y_pred - y))
            self.w = self.learning_rate * gred
    def _fit_sgred(self,x,y):
        np.random.seed(self.seed)
        
        Xb = np.c_[np.ones((x.shape[0], 1)), x]
        self.w = np.zeros(Xb.shape[1], dtype=np.float64)
        n = Xb.shape[0]

        for x in range(self.iteration):
            indices = np.random.permutation(n)
            y_shuffled = y[indices]
            Xb_shuffled = Xb[indices]
        for i in range(0, n, self.batch_size):
                X_batch = Xb_shuffled[i: i+self.batch_size]
                y_batch = y_shuffled[i: i+self.batch_size]
                y_pred = X_batch @ self.w
                grad = (2 / len(X_batch)) * (X_batch.T @ (y_pred - y_batch))
                self.w -= self.learning_rate * grad

    def fit(self, X, y):
        if self.method == 'analytic':
            self.linearregression(X, y)
        elif self.method == 'gd':
            self.fit_gb(X, y)
        elif self.method == 'sgd':
            self._fit_sgred(X, y)

    def predict(self,X):
        X = np.c_[np.ones((X.shape[0], 1)), X]
        return X.dot(self.w)
      

In [580]:
lr3  = regression(method='sgd')
lr3.fit(x_train,y_tarin)


KeyError: '[1270, 48361, 10115, 5098, 28942, 16139, 41754, 38872, 41147, 3107, 46657, 32999, 23487, 34711, 24352, 22202, 3435, 22353, 15416, 47795, 7419, 34628, 37237, 43600, 4089, 22983, 27299, 27771, 39060, 34207, 17479, 29431, 33353, 13622, 10457, 17344, 30315, 22761, 34515, 37346, 39823, 2553, 27380, 22092, 22921, 31607, 36684, 9463, 16179, 3630, 39965, 20229, 29701, 3078, 46968, 1098, 29266, 30681, 16201, 3992, 6872, 10840, 8298, 21309, 21439, 3343, 35318, 47411, 16448, 33929, 24619, 47473, 18713, 7, 45861, 31795, 3525, 7746, 29260, 41424, 29563, 43088, 33341, 36824, 5033, 35208, 31619, 11725, 32081, 34810, 18022, 16099, 19015, 23689, 18929, 31743, 29028, 12684, 3203, 17334, 6726, 5004, 9378, 43105, 11922, 29473, 3366, 4707, 46455, 5896, 43951, 9217, 18974, 41613, 20469, 23784, 42722, 11294, 16699, 35305, 40356, 36623, 24470, 42580, 38978, 4238, 25429, 46411, 2657, 33857, 17210, 15350, 18591, 16319, 20006, 29189, 35113, 24333, 44362, 39491, 10461, 42946, 1575, 40731, 38342, 16391,

In [ ]:
lr2 = regression(method='gd')
lr2.fit(x_train,y_tarin)
print(lr2.w)

[ -7.1813823   -9.57570946 -12.98029995  -4.12044017  -3.60098964
  -3.52884143  -3.32103543  -3.49365286  -3.34657804  -2.85502888
  -2.60739469  -2.26600357  -1.29633733  -1.57481697  -1.09285193
  -0.91669477  -0.94848614  -0.71595094  -0.53241258  -0.48872002
  -0.36739656  -0.42217389  -0.41656548  -0.30175215]


In [ ]:
lr1 = regression(method='analytic')
lr1.fit(x_train,y_tarin)
print(lr1.w)

[ 594.16774449 1575.61950639  463.19262189  225.94956994 -171.13946822
  -82.29562883  155.22419075  619.24285788  149.26531835 -164.70166353
 -219.63365493  243.62361178  -25.48940922  456.82108237 -105.10019517
  -91.9596842   121.56421258 -228.87031369  -38.52624836   34.8774644
 -363.80845622 -114.37794051  175.4540985    36.24622721]


In [ ]:
model = regression()
model.fit(x_train, y_tarin)
model.w

KeyError: '[1270, 48361, 10115, 5098, 28942, 16139, 41754, 38872, 41147, 3107, 46657, 32999, 23487, 34711, 24352, 22202, 3435, 22353, 15416, 47795, 7419, 34628, 37237, 43600, 4089, 22983, 27299, 27771, 39060, 34207, 17479, 29431, 33353, 13622, 10457, 17344, 30315, 22761, 34515, 37346, 39823, 2553, 27380, 22092, 22921, 31607, 36684, 9463, 16179, 3630, 39965, 20229, 29701, 3078, 46968, 1098, 29266, 30681, 16201, 3992, 6872, 10840, 8298, 21309, 21439, 3343, 35318, 47411, 16448, 33929, 24619, 47473, 18713, 7, 45861, 31795, 3525, 7746, 29260, 41424, 29563, 43088, 33341, 36824, 5033, 35208, 31619, 11725, 32081, 34810, 18022, 16099, 19015, 23689, 18929, 31743, 29028, 12684, 3203, 17334, 6726, 5004, 9378, 43105, 11922, 29473, 3366, 4707, 46455, 5896, 43951, 9217, 18974, 41613, 20469, 23784, 42722, 11294, 16699, 35305, 40356, 36623, 24470, 42580, 38978, 4238, 25429, 46411, 2657, 33857, 17210, 15350, 18591, 16319, 20006, 29189, 35113, 24333, 44362, 39491, 10461, 42946, 1575, 40731, 38342, 16391,

In [ ]:
ytrue = np.array(y_tarin).reshape((-1,1))
ypred = np.array(y_test).reshape((-1,1))


In [ ]:
def r2(y_true ,y_pred):
    sse_re = np.sum((y_true - y_pred)** 2)
    sse_tot = np.sum((y_true - np.mean(y_true))** 2)
    if sse_tot == 0 :
        return 1
    else :
        return 1 - (sse_re / sse_tot)
def rmse(ytrue, ypred):
    n = len(ytrue)
    mse = 1/n * np.sum((ytrue - ypred)**2)
    return np.sqrt(mse)
def mae(ytrue,ypred):
    n  = len(ytrue )
    mae = np.sum(np.abs(ytrue - ypred)) / n
    return mae

In [ ]:
y_train_pred = model.predict(x_train)
y_test_pred = model.predict(X_test)

In [ ]:
print(f"R2: {r2(y_true=y_tarin, y_pred=y_train_pred)}")
print(f"MAE: {mae(y_tarin, y_train_pred)}")
print(f"RMSE: {rmse(y_tarin, y_train_pred)}")

R2: -4.905807826642819
MAE: 3538.6368879059096
RMSE: 3882.5797943859275
